In [1]:
import sys
import os
import logging
import time
from time import sleep
from typing import Dict, List

# Add the client/python directory to the Python path
sys.path.append(os.path.abspath('../client/python'))

from pump_comm import SyringePumpController

# Configure logging with timestamp
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

In [5]:
# Example #1: Pump Status

ctrl = SyringePumpController('COM7')

print(ctrl.get_pump_status('A'))
print(ctrl.get_pump_status('B'))
print(ctrl.get_pump_status('C'))
print(ctrl.get_pump_status('D'))

ctrl.close()

PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
PUMP=B FLOW=1000.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
PUMP=C FLOW=2000.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
PUMP=D FLOW=3000.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON


In [20]:
# Example #2: Pump configuration

ctrl = SyringePumpController('COM7')

pump = 'B' # Select a pump
status = ctrl.get_pump_status(pump)
print(f"Initial status:\n{status}")

# Make configuration changes
ctrl.set_flow(pump, 1000)
ctrl.set_unit(pump, 'UL/HR')
ctrl.set_diameter(pump, 8.17)
ctrl.set_direction(pump, 'INFUSE')
ctrl.set_gearbox(pump, '1:1')
ctrl.set_microstep(pump, '1/16')
ctrl.set_threadrod(pump, '1-START')

# Get the updated status
status = ctrl.get_pump_status(pump)
print(f"New status:\n{status}")

ctrl.close()

Initial status:
PUMP=B FLOW=250.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON
New status:
PUMP=B FLOW=1000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON


In [16]:
# Example #3: Controlling two pumps

ctrl = SyringePumpController('COM7')

ctrl.set_flow('A', 2000)
ctrl.set_flow('B', 1000)

period = 10 # both pumps run X s

pumps = ['A', 'B']

for p in pumps:
    ctrl.set_unit(p, 'UL/HR')
    ctrl.set_state(p, 'RUN')
    ctrl.set_direction(p, 'INFUSE')
    print(ctrl.get_pump_status(p))
    
print(f"\nRunning pumps for {period} seconds...\n")

time.sleep(period)                     

for p in pumps:
    ctrl.set_state(p, 'STOP')
    print(ctrl.get_pump_status(p))

print("\nPumps stopped")

ctrl.close()

PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
PUMP=B FLOW=1000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON

Running pumps for 10 seconds...

PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
PUMP=B FLOW=1000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON

Pumps stopped


In [17]:
# Example #4: Controlling two pumps with logs

ctrl = SyringePumpController('COM7')

ctrl.set_flow('A', 2000)
ctrl.set_flow('B', 1000)

period = 10 # both pumps run X s

pumps = ['A', 'B']

for p in pumps:
    ctrl.set_unit(p, 'UL/HR')
    ctrl.set_state(p, 'RUN')
    ctrl.set_direction(p, 'INFUSE')
    status = ctrl.get_pump_status(p)
    logging.info(f"Pump {p} initial status: {status}")
    
print(f"\nRunning pumps for {period} seconds...\n")

time.sleep(period)                     

for p in pumps:
    ctrl.set_state(p, 'STOP')
    status = ctrl.get_pump_status(p)
    logging.info(f"Pump {p} final status: {status}")

print("\nPumps stopped")

ctrl.close()

2025-09-29 16:27:47 Pump A initial status: PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
2025-09-29 16:27:47 Pump B initial status: PUMP=B FLOW=1000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON



Running pumps for 10 seconds...



2025-09-29 16:27:57 Pump A final status: PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
2025-09-29 16:27:57 Pump B final status: PUMP=B FLOW=1000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON



Pumps stopped


In [18]:
# Example #5: Controlling pumps in cycles, with configurable run time, pause time, and number of repeats

def run_pump_cycle(pump, settings: Dict[str, Dict[str, any]], run_time, 
                  pause_time, num_repeats) -> None:
    """
    Run pumps in cycles with configurable timing and pump settings
    
    Args:
        pump: Pump controller instance
        settings: Dictionary of pump settings. Example:
            {
                'A': {
                    'flow_rate': 1000,  # in µL/h
                    'diameter': 4.78,   # in mm
                    'unit': 'UL/HR'     # flow rate unit
                }
            }
        run_time: Time to run pumps in seconds
        pause_time: Time to pause between runs in seconds
        num_repeats: Number of cycles to run
    """
    # Print initial status
    print("\n" + "="*50)
    print(f"Starting pump cycle program")
    print(f"Number of cycles: {num_repeats}")
    print(f"Run time per cycle: {run_time} seconds")
    print(f"Pause time between cycles: {pause_time} seconds")
    print("="*50 + "\n")

    # Configure all the pumps
    for p, config in settings.items():
        try:
            # Set pump parameters
            pump.set_diameter(p, config['diameter'])
            pump.set_unit(p, config['unit'])
            pump.set_flow(p, config['flow_rate'])
        except Exception as e:
            logging.error(f"Error starting pump {p}: {e}")
            return
    
    for cycle in range(1, num_repeats + 1):

        # Print cycle header with clear separation
        print("\n" + "-"*50)
        print(f"🚀 CYCLE {cycle}/{num_repeats} - STARTING")
        print("-"*50 + "\n")
               
        # Start all pumps with their settings
        for p, config in settings.items():
            pump.set_direction(p, 'INFUSE')
            pump.set_state(p, 'RUN')
            status = pump.get_pump_status(p)
            print(f"✅ Pump {p} started with: {status}")
        
        # Run for specified time
        print(f"\n⏳ Running for {run_time} seconds...")
        time.sleep(run_time)
        
        # Stop all pumps
        print("\n🛑 Stopping all pumps...\n")
        
        for p in settings:
            try:
                pump.set_state(p, 'STOP')
                status = pump.get_pump_status(p)
                print(f"⏹️ Pump {p} stopped: {status}")
            except Exception as e:
                logging.error(f"Error stopping pump {p}: {e}")
        
        # Pause between cycles (except after last cycle)
        if cycle < num_repeats:
            print(f"\n⏸️ Pausing for {pause_time} seconds...")
            time.sleep(pause_time)
    
    print("\n" + "="*50)
    print("✅ ALL CYCLES COMPLETED SUCCESSFULLY")
    print("="*50 + "\n")

def main():
    # Configuration
    ctrl = SyringePumpController('COM7')
    
    # Updated settings with flow, diameter, and unit
    PUMP_SETTINGS = {
        'A': {
            'flow_rate': 2000,  # in µL/h
            'diameter': 8.17,   # in mm
            'unit': 'UL/HR'     # flow rate unit
        },
        'B': {
            'flow_rate': 1000,
            'diameter': 4.78,
            'unit': 'UL/HR'
        }
    }
    
    RUN_TIME = 10.0    # Run for X seconds
    PAUSE_TIME = 10.0  # Pause for Y seconds between runs
    NUM_REPEATS = 3    # Number of cycles to run
    
    try:        
        # Run the cycle
        run_pump_cycle(
            ctrl,
            settings=PUMP_SETTINGS,
            run_time=RUN_TIME,
            pause_time=PAUSE_TIME,
            num_repeats=NUM_REPEATS
        )
            
    except Exception as e:
        logging.error(f"An error occurred: {e}")
    finally:
        ctrl.close()
        print("Script completed")

if __name__ == "__main__":
    main()


Starting pump cycle program
Number of cycles: 3
Run time per cycle: 10.0 seconds
Pause time between cycles: 10.0 seconds


--------------------------------------------------
🚀 CYCLE 1/3 - STARTING
--------------------------------------------------

✅ Pump A started with: PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
✅ Pump B started with: PUMP=B FLOW=1000.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON

⏳ Running for 10.0 seconds...

🛑 Stopping all pumps...

⏹️ Pump A stopped: PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
⏹️ Pump B stopped: PUMP=B FLOW=1000.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON

⏸️ Pausing for 10.0 seconds...

--------------------------------------------------
🚀 CYCLE 2/3 - STARTING
------------------

In [19]:
# Example #6: Controlling pumps with flexible sequence definition

def run_pump_sequence(pump, sequence: List[Dict]) -> None:
    """
    Run pumps through a sequence of operations with different flow rates
    
    Args:
        pump: Pump controller instance
        sequence: List of operations, where each operation is a dict with:
            - 'duration': Time in seconds to run this step
            - 'pumps': Dict of pump settings, e.g. {
                'A': {
                    'flow_rate': 1000,  # in µL/h
                    'diameter': 8.17,   # in mm
                    'unit': 'UL/HR'     # flow rate unit
                }
            }
            - 'description': Optional description of the step
    """
    # Print initial status
    print("\n" + "="*50)
    print(f"Starting pump sequence program")
    print(f"Number of steps: {len(sequence)}")
    print("="*50 + "\n")

    # Configure all pumps that will be used
    used_pumps = set()
    for step in sequence:
        used_pumps.update(step['pumps'].keys())

    for p in used_pumps:
        try:
            config = next(step['pumps'][p] for step in sequence if p in step['pumps'])
            pump.set_diameter(p, config['diameter'])
            pump.set_unit(p, config['unit'])
        except Exception as e:
            print(f"❌ Error configuring pump {p}: {e}")
            return
    
    for step_num, step in enumerate(sequence, 1):
        duration = step['duration']
        pump_settings = step.get('pumps', {})
        desc = step.get('description', f"Step {step_num}")
        
        # Print step header
        print("\n" + "-"*50)
        print(f"🚀 STEP {step_num}/{len(sequence)}: {desc}")
        print("-"*50)

        if pump_settings:
            print("\nStarting pumps:")
            for p, config in pump_settings.items():
                try:
                    pump.set_flow(p, config['flow_rate'])
                    pump.set_direction(p, 'INFUSE')
                    pump.set_state(p, 'RUN')
                    status = pump.get_pump_status(p)
                    print(f"✅ Pump {p}: {status}")
                except Exception as e:
                    print(f"❌ Error starting pump {p}: {e}")
            
            # Run for specified duration
            print(f"\n⏳ Running for {duration} seconds...")
            time.sleep(duration)

            # Stop all pumps that were running in this step
            print("\nStopping pumps:")
            for p in pump_settings:
                try:
                    pump.set_state(p, 'STOP')
                    print(f"⏹️ Pump {p} stopped: {pump.get_pump_status(p)}")
                except Exception as e:
                    print(f"❌ Error stopping pump {p}: {e}")
        else:
            print("\n⏸️ No pumps active in this step")
            time.sleep(duration)
    
    print("\n" + "="*50)
    print("✅ SEQUENCE COMPLETED SUCCESSFULLY")
    print("="*50 + "\n")

def main():
    
    # Configuration
    ctrl = SyringePumpController('COM7')
    
    # Define your sequence of operations
    SEQUENCE = [
        {
            'description': "Initial ramp up",
            'duration': 10,  # seconds
            'pumps': {
                'A': {
                    'flow_rate': 2000,
                    'diameter': 8.17,
                    'unit': 'UL/HR'
                },
                'B': {
                    'flow_rate': 1000,
                    'diameter': 4.78,
                    'unit': 'UL/HR'
                }
            }
        },
        {
            'description': "Pause all pumps",
            'duration': 5,  # Pause time
            'pumps': {}      # Empty dict means all pumps off
        },
        {
            'description': "Medium flow rate",
            'duration': 10,
            'pumps': {
                'A': {
                    'flow_rate': 1000,
                    'diameter': 8.17,
                    'unit': 'UL/HR'
                },
                'B': {
                    'flow_rate': 500,
                    'diameter': 4.78,
                    'unit': 'UL/HR'
                }
            }
        },
        {
            'description': "Pause all pumps",
            'duration': 5,  # Pause time
            'pumps': {}      # Empty dict means all pumps off
        },
        {
            'description': "Last flow rate",
            'duration': 10,
            'pumps': {
                'A': {
                    'flow_rate': 500,
                    'diameter': 8.17,
                    'unit': 'UL/HR'
                },
                'B': {
                    'flow_rate': 250,
                    'diameter': 4.78,
                    'unit': 'UL/HR'
                }
            }
        }
    ]
    
    try:    
        # Run the sequence
        run_pump_sequence(ctrl, SEQUENCE)
            
    except Exception as e:
        logging.error(f"An error occurred: {e}")
    finally:
        ctrl.close()
        print("Script completed")

if __name__ == "__main__":
    main()


Starting pump sequence program
Number of steps: 5


--------------------------------------------------
🚀 STEP 1/5: Initial ramp up
--------------------------------------------------

Starting pumps:
✅ Pump A: PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
✅ Pump B: PUMP=B FLOW=1000.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=RUN UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON

⏳ Running for 10 seconds...

Stopping pumps:
⏹️ Pump A stopped: PUMP=A FLOW=2000.00 DIAMETER=8.17 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/16 ROD=1-START ENABLE=ON
⏹️ Pump B stopped: PUMP=B FLOW=1000.00 DIAMETER=4.78 DIRECTION=INFUSE STATE=STOP UNIT=UL/HR GEARBOX=1:1 MICROSTEP=1/8 ROD=1-START ENABLE=ON

--------------------------------------------------
🚀 STEP 2/5: Pause all pumps
--------------------------------------------------

⏸️ No pumps active in this step

-----------------------------------------------